### Import Libraries

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


### Import Data

In [0]:
df_drivers = spark.read.csv('s3://columbia-gr5069-main/raw/drivers.csv', header=True)
df_drivers = df_drivers.toPandas()
df_races = spark.read.csv('s3://columbia-gr5069-main/raw/races.csv', header=True)
df_races = df_races.toPandas() 
df_results = spark.read.csv('s3://columbia-gr5069-main/raw/results.csv', header=True)
df_results = df_results.toPandas()

In [0]:
df_drivers.head()

In [0]:
df_races.head()

In [0]:
df_results.head()

### Preprocessing 

Get data per driver.

In [0]:
new_drivers = pd.DataFrame({
    'driverId': df_drivers['driverId'],
    'nationality': df_drivers['nationality'],
    'name': df_drivers['forename'] + ' ' + df_drivers['surname']
})

new_drivers.head()

In [0]:
new_drivers = pd.merge(new_drivers, df_results[['driverId', 'raceId', 'points', 'laps', 'milliseconds']], on='driverId', how='left')

In [0]:
new_drivers = pd.merge(new_drivers, df_races[['raceId', 'year']], on='raceId', how='left')
new_drivers.head()

In [0]:
new_drivers = pd.merge(new_drivers, df_results[['raceId', 'position']],on='raceId', how='left')
new_drivers.head()

In [0]:
new_drivers['points'] = pd.to_numeric(new_drivers['points'], errors='coerce').fillna(0).astype(int)
new_drivers['laps'] = pd.to_numeric(new_drivers['laps'], errors='coerce').fillna(0).astype(int)
new_drivers['milliseconds'] = pd.to_numeric(new_drivers['milliseconds'], errors='coerce').fillna(0).astype(int)
new_drivers['year'] = pd.to_numeric(new_drivers['year'], errors='coerce').fillna(0).astype(int)
new_drivers['position'] = pd.to_numeric(new_drivers['position'], errors='coerce').fillna(0).astype(int)

In [0]:
new_drivers

In [0]:
final_driver_data = new_drivers.groupby('driverId')['raceId'].nunique().reset_index()
final_driver_data = final_driver_data.rename(columns={'raceId': 'total_number_of_races'})
final_driver_data = new_drivers.groupby('driverId')['year'].nunique().reset_index()
final_driver_data = final_driver_data.rename(columns={'year': 'total_number_of_year_in_f1'})
final_driver_data['avg_points_per_race'] = new_drivers.groupby('driverId')['points'].mean().reset_index()['points']
final_driver_data['avg_laps_per_race'] = new_drivers.groupby('driverId')['laps'].mean().reset_index()['laps']
final_driver_data['avg_milliseconds_per_race'] = new_drivers.groupby('driverId')['milliseconds'].mean().reset_index()['milliseconds']


In [0]:
races_won = pd.DataFrame(new_drivers[new_drivers['position'] == 1].groupby('driverId').size())
final_driver_data = pd.merge(final_driver_data, races_won, on='driverId', how='left')

final_driver_data = final_driver_data.rename(columns={0: 'total_number_of_races_won'})

In [0]:
final_driver_data = final_driver_data.applymap(lambda x: int(x) if isinstance(x, (int, float)) else x)
final_driver_data


### Train Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    final_driver_data.drop(["driverId", "total_number_of_races_won"], axis=1),
    final_driver_data["total_number_of_races_won"].values.ravel(),
    random_state=42
)

### Linear Regression without Hyper Tuning